# 采购入库数据清洗

按异常类型逐项过滤查看。每个小节只回答一件事：**这类异常有多少行、长什么样、怎么处理**。

口径：`read_inbound("2026-09")`，41,849 行 × 51 列。
**退货判定只看 `数量 < 0`**，`出入库类型名称` 不参与判定、清洗时丢弃。
清洗规则见 [采购入库](../docs/数据接入/采购入库.md) 第五节；本 notebook 只做查看和验证，不改归档数据。

| 小节 | 看什么 |
| --- | --- |
| 一 | 退货判定 + 单价合并 + 金额勾稽 |
| 二 | 退货行明细与类型字段的交叉核对 |
| 三 | 单价为空 / 为 0 |
| 四 | 金额为 0 / 为负 / 成本倒挂 |
| 五 | 完全重复行 |
| 六 | 字段空值 |
| 七 | 枚举脏值 |
| 八 | 单据编码格式 |
| 九 | 异常清单汇总 |
| 十 | 清洗预演：产出 44 列底表 |

In [1]:
import pandas as pd

import read_data

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 46)

df = read_data.read_inbound("2026-09")

# 退货 = 数量为负，不看出入库类型名称（该列清洗时丢弃）
df["是否退货"] = df["数量"] < 0

print(f"{len(df):,} 行 × {df.shape[1] - 1} 列")
print(f"退货（数量<0）：{df['是否退货'].sum()} 行 | 入库：{(~df['是否退货']).sum()} 行")
print(df["数据来源系统"].value_counts().to_string())

41,849 行 × 51 列
退货（数量<0）：160 行 | 入库：41689 行
数据来源系统
ebs    21452
nc     20397


## 一、退货判定与单价合并、金额勾稽

两个系统把退货单价放在不同列：ebs 只有 `购退单价`（`采购单价` 全为 0），nc 只有 `采购单价`。
不合并的话按列聚合会各丢一半，而且 ebs 的 `单价 × 数量 ≠ 金额`。

合并规则：`数量 < 0` 且 `数据来源系统 == "ebs"` 且 `购退单价` 非空 → 用 `购退单价` 覆盖 `采购单价`，
不含税同理。

In [2]:
取退货价 = df["是否退货"] & (df["数据来源系统"] == "ebs") & df["购退单价"].notna()

clean = df.copy()
clean["采购单价"] = clean["采购单价"].mask(取退货价, clean["购退单价"])
clean["采购不含税单价"] = clean["采购不含税单价"].mask(取退货价, clean["购退不含税单价"])
clean = clean.drop(columns=["购退单价", "购退不含税单价"])

calc = clean["采购单价"].fillna(0) * clean["数量"].fillna(0) / 10000
勾稽 = (calc - clean["采购金额(万)"].fillna(0)).abs().le(0.01)

print(f"被 ebs 退货价覆盖：{取退货价.sum()} 行")
print(f"勾稽 单价×数量=金额：{勾稽.sum():,} / {len(clean):,} = {勾稽.mean():.2%}")
print(f"列数：{df.shape[1] - 1} → {clean.shape[1] - 1}（均不含派生的「是否退货」）")
print(f"合并后单价仍为 0：{(clean['采购单价'].fillna(0) == 0).sum():,} 行，金额合计 "
      f"{clean.loc[clean['采购单价'].fillna(0) == 0, '采购金额(万)'].sum():,.2f} 万")

clean.loc[~勾稽, ["出入库日期", "物料名称", "数量", "采购单价", "采购金额(万)"]]

被 ebs 退货价覆盖：90 行
勾稽 单价×数量=金额：41,849 / 41,849 = 100.00%
列数：51 → 49（均不含派生的「是否退货」）
合并后单价仍为 0：4,566 行，金额合计 0.00 万


,出入库日期,物料名称,数量,采购单价,采购金额(万)


## 二、退货行明细与类型字段的交叉核对

按数量符号判定是 160 行，按 `出入库类型名称` 判定是 157 行，差的 3 行是 nc 单据
`CR2015260900230` 的红冲行——类型标着「普通采购入库」，数量却是负。按符号判定后自动归入退货，
不需要特判。

In [3]:
类型退货 = df["出入库类型名称"].isin(["退回供应商", "采购退货"])

print(f"按数量符号：{df['是否退货'].sum()} 行 | 按类型字段：{类型退货.sum()} 行")
print(pd.crosstab(df["出入库类型名称"], df["是否退货"], rownames=["类型"], colnames=["数量<0"]).to_string())

print("\n符号判为退货、类型却不是退货的行：")
df.loc[df["是否退货"] & ~类型退货,
       ["出入库日期", "数据来源系统", "出入库类型名称", "单据编码", "物料名称", "数量", "采购金额(万)"]]

按数量符号：160 行 | 按类型字段：157 行
数量<0    False  True 
类型                  
PO接收    21362      0
普通采购入库  20327      3
退回供应商       0     90
采购退货        0     67

符号判为退货、类型却不是退货的行：


,出入库日期,数据来源系统,出入库类型名称,单据编码,物料名称,数量,采购金额(万)
40545,2026-09-29,nc,普通采购入库,CR2015260900230,尿核基质蛋白22(NMP22)检测试剂盒 NMP22 BladderChek Test,-20.0,-7.956
40687,2026-09-29,nc,普通采购入库,CR2015260900230,一次性塑料吸管,-24.0,-0.165
41242,2026-09-29,nc,普通采购入库,CR2015260900230,NTMT1和 MAP3K14-AS1基因甲基化检测试剂盒,-20.0,-11.934


## 三、单价为空 / 为 0

- `采购单价` 空 90 行：都是 ebs 的退货行，已由 `购退单价` 补齐。
- `采购单价` 为 0 共 4,655 行：<u>不等于缺失</u>，是赠品、样品或未计价的入库。
- **单价为 0 但金额非 0** 的 89 行最需要看——说明金额不是按这一行单价算出来的，
  合并后这批会自动消失。

In [4]:
单价空 = df[df["采购单价"].isna()]
单价零 = df[df["采购单价"].fillna(0) == 0]
零价有额 = df[(df["采购单价"].fillna(0) == 0) & (df["采购金额(万)"].fillna(0) != 0)]

print(f"采购单价 空 {len(单价空)} | =0 {len(单价零)} | =0 但金额非 0 {len(零价有额)} 行")
print(f"零价有额 按退货：{零价有额['是否退货'].value_counts().to_dict()}")
print(f"零价有额 按系统：{零价有额['数据来源系统'].value_counts().to_dict()}")

零价有额[["出入库日期", "数据来源系统", "物料名称", "数量", "采购单价", "购退单价",
         "采购金额(万)", "成本单价"]]

采购单价 空 90 | =0 4655 | =0 但金额非 0 89 行
零价有额 按退货：{True: 89}
零价有额 按系统：{'ebs': 89}


,出入库日期,数据来源系统,物料名称,数量,采购单价,购退单价,采购金额(万),成本单价
1184,2026-09-01,ebs,乙型肝炎病毒核心抗体校准品（定性）,-1.0,NaN,103.00,-0.010300,91.150442
1748,2026-09-01,ebs,乙型肝炎病毒 e 抗体校准品（定性）,-1.0,NaN,103.00,-0.010300,91.150442
3673,2026-09-01,ebs,白蛋白测定试剂盒（）,-1.0,NaN,22.97,-0.002297,20.327434
8386,2026-09-04,ebs,总三碘甲状腺原氨酸(T3)测定试剂盒（）,-3.0,NaN,334.75,-0.100425,296.238938
8685,2026-09-04,ebs,人类免疫缺陷病毒抗原抗体(HIV)测定试剂盒,-4.0,NaN,767.97,-0.307188,679.619469
...,...,...,...,...,...,...,...,...
40585,2026-09-29,ebs,缓冲液（特定蛋白分析仪用(CRP)）,-10.0,NaN,105.00,-0.105000,92.920354
40842,2026-09-29,ebs,尿液分析试纸条（适用于 FUS-3000Plus/MUS-3600/MUS-9600）,-63.0,NaN,850.00,-5.355000,752.212389
41458,2026-09-30,ebs,一次性使用拭子（塑棒）,-5.0,NaN,540.00,-0.270000,477.876106
41544,2026-09-30,ebs,B 群链球菌显色平板,-4.0,NaN,173.88,-0.069552,153.876106


## 四、金额为 0 / 为负 / 成本倒挂

注意**符号陷阱**：退货行金额为负，直接用 `成本金额 > 采购金额` 会把 155 行误判成倒挂，按绝对值比才对。

In [5]:
for 列 in ["采购金额(万)", "成本金额(万)", "采购不含税金额(万)"]:
    s = df[列].fillna(0)
    print(f"{列:<16} =0 {(s == 0).sum():>6,} | <0 {(s < 0).sum():>5,} | 合计 {s.sum():>12,.2f} 万")

原值倒挂 = df["成本金额(万)"].fillna(0) > df["采购金额(万)"].fillna(0)
绝对值倒挂 = df["成本金额(万)"].fillna(0).abs() > df["采购金额(万)"].fillna(0).abs() + 1e-9
print(f"\n成本 > 采购（原值比较）：{原值倒挂.sum()} 行，其中退货 {df.loc[原值倒挂, '是否退货'].sum()}")
print(f"成本 > 采购（绝对值比较）：{绝对值倒挂.sum()} 行 ← 真实倒挂")

df.loc[原值倒挂, ["是否退货", "数量", "采购单价", "成本单价", "采购金额(万)", "成本金额(万)"]].head(5)

采购金额(万)          =0  4,566 | <0   156 | 合计    15,374.01 万
成本金额(万)          =0  4,579 | <0   156 | 合计    13,531.96 万
采购不含税金额(万)       =0  4,566 | <0   156 | 合计    13,620.30 万

成本 > 采购（原值比较）：155 行，其中退货 155
成本 > 采购（绝对值比较）：0 行 ← 真实倒挂


,是否退货,数量,采购单价,成本单价,采购金额(万),成本金额(万)
70,True,-1.0,18.7,16.550000,-0.00187,-0.001655
202,True,-21.1,1800.0,1592.920000,-3.79800,-3.361062
400,True,-100.0,278.0,246.020000,-2.78000,-2.460177
1184,True,-1.0,NaN,91.150442,-0.01030,-0.009115
1748,True,-1.0,NaN,91.150442,-0.01030,-0.009115


## 五、完全重复行

同一单据 + 同一物料 + 同一批次 + 同数量 + 同金额的行出现两次，是文件内完全重复，不是多明细。
去重会直接影响金额合计，需要业务确认是上游重复导出还是确有两笔。

In [6]:
重复 = df[df.duplicated(keep="first")]
去重 = df.drop_duplicates(keep="first")

print(f"多余重复行 {len(重复)} | 去重后 {len(去重):,} / {len(df):,}")
print(f"重复行按系统：{重复['数据来源系统'].value_counts().to_dict()}")
print(f"\n采购金额：原 {df['采购金额(万)'].sum():,.2f} 万 → 去重后 {去重['采购金额(万)'].sum():,.2f} 万"
      f"（差 {重复['采购金额(万)'].sum():,.2f} 万）")

重复[["单据编码", "数据来源系统", "物料编码", "物料名称", "批次号", "数量", "采购金额(万)"]].head(10)

多余重复行 209 | 去重后 41,640 / 41,849
重复行按系统：{'ebs': 176, 'nc': 33}

采购金额：原 15,374.01 万 → 去重后 15,316.96 万（差 57.05 万）


,单据编码,数据来源系统,物料编码,物料名称,批次号,数量,采购金额(万)
313,30040106473 - 12,ebs,11063135,测序反应通用试剂盒-DNB 制备试剂试剂盒和测序试剂盒(1/2),MG2607005,4.0,1.200000
1247,30040106473 - 12,ebs,11063135,测序反应通用试剂盒-DNB 制备试剂试剂盒和测序试剂盒(1/2),MG2607005,4.0,1.200000
1377,CR2009260900026,nc,11020969,促肾上腺皮质激素测定试剂盒（）,20260401,1.0,0.093241
1459,30040106550 - 29,ebs,11079241,尿素测定试剂盒（）,0426121,2.0,0.009654
2026,30040106473 - 12,ebs,11063135,测序反应通用试剂盒-DNB 制备试剂试剂盒和测序试剂盒(1/2),MG2607005,4.0,1.200000
2179,CR2052260900082,nc,11027581,人类免疫缺陷病毒抗原抗体非定值质控品,20260201,2.0,0.000000
2753,CR2033260900023,nc,11055121,抗甲状腺球蛋白抗体检测试剂盒 Elecsys Anti-Tg（cobas e 411...,96416701,1.0,0.105206
2770,CR2003260900049,nc,11026201,抗环瓜氨酸肽(CCP)抗体酶联免疫检测试剂盒,EC260501,3.0,0.139050
3015,30040106473 - 12,ebs,11063135,测序反应通用试剂盒-DNB 制备试剂试剂盒和测序试剂盒(1/2),MG2607005,4.0,1.200000
3312,CR2012260900016,nc,11020986,EB 病毒衣壳抗原 IgG 抗体检测试剂盒（）,20260601,2.0,0.000000


## 六、字段空值

只有 9 个字段有空值。`货号` 的 14.75% 是唯一的量级缺失，`多组织名称` 空 95.9%（清洗时已丢弃）。

In [7]:
(
    pd.DataFrame({"空行数": df.isna().sum()})
    .assign(空值率=lambda x: (x["空行数"] / len(df)).map("{:.2%}".format))
    .query("空行数 > 0")
    .sort_values("空行数", ascending=False)
)

,空行数,空值率
购退单价,41759,99.78%
多组织名称,40134,95.90%
货号,6171,14.75%
财务内部外供应商类型,689,1.65%
采购类型名称,220,0.53%
生产地址,91,0.22%
采购单价,90,0.22%
生产国,79,0.19%
仓库类型,4,0.01%


## 七、枚举脏值

四个需要归并的取值：`arr` 是单位脏值，`科研产品2` 并入科研产品，两个 `无分类` 置空，
`其他` 保留原值只在报表层合并。

In [8]:
for 列, 值, 处理 in [
    ("主单位", "arr", "置空"),
    ("产品类型", "科研产品2", "并入「科研产品」"),
    ("三级分类", "其他", "保留，报表层合并"),
    ("四级分类", "无分类", "置空"),
    ("五级分类", "无分类", "置空"),
]:
    print(f"{列:<8} = {值:<6} {(df[列] == 值).sum():>6,} 行   → {处理}")

df.loc[df["主单位"] == "arr", ["出入库日期", "物料名称", "主单位", "数量", "采购金额(万)"]]

主单位      = arr         4 行   → 置空
产品类型     = 科研产品2       4 行   → 并入「科研产品」
三级分类     = 其他          4 行   → 保留，报表层合并
四级分类     = 无分类     3,765 行   → 置空
五级分类     = 无分类     3,765 行   → 置空


,出入库日期,物料名称,主单位,数量,采购金额(万)
20921,2026-09-12,革兰染色液（搭配染色机）,arr,1.0,0.080000
29022,2026-09-18,蜡块柜,arr,2.0,0.285000
40597,2026-09-29,杯架组件,arr,25.0,0.540275
41078,2026-09-29,杯架组件,arr,5.0,0.108055


## 八、单据编码格式

两种格式完全按系统区分：ebs 是 `3070005745 - 3`（带空格和行号），nc 是 `CR2001260900358`。
要拆主单号和行号，只对 ebs 侧做。

In [9]:
编码 = df["单据编码"].astype(str)
含行号 = 编码.str.contains(" - ", regex=False)

print(pd.DataFrame({
    "行数": df.groupby("数据来源系统").size(),
    "含「 - 」": 含行号.groupby(df["数据来源系统"]).sum(),
    "前缀 CR": 编码.str.startswith("CR").groupby(df["数据来源系统"]).sum(),
}).to_string())
print()
print("ebs 样例：", 编码[含行号].head(3).tolist())
print("nc  样例：", 编码[~含行号].head(3).tolist())

           行数  含「 - 」  前缀 CR
数据来源系统                      
ebs     21452   21452      0
nc      20397       0  20397

ebs 样例： ['3070005745 - 3', '30020033429 - 9', '30040106600 - 3']
nc  样例： ['CR2006260900010', 'CR2052260900049', 'CR2007260900034']


## 九、异常清单汇总

一张表看全部：行数、占比、处置。需要业务确认的只有「重复行」这一项。

In [10]:
项 = [
    ("退货行（数量<0）", df["是否退货"], "单价按系统合并，见一"),
    ("退货且 ebs，单价由购退单价覆盖", 取退货价, "已合并"),
    ("采购单价为空", df["采购单价"].isna(), "合并后有值"),
    ("采购单价为 0", df["采购单价"].fillna(0) == 0, "赠品/未计价，打标记不删"),
    ("单价为 0 但金额非 0", (df["采购单价"].fillna(0) == 0) & (df["采购金额(万)"].fillna(0) != 0), "合并后转为 0 行"),
    ("合并后单价仍为 0", clean["采购单价"].fillna(0) == 0, "金额也全为 0，无影响"),
    ("采购金额 < 0", df["采购金额(万)"].fillna(0) < 0, "退货行，正常"),
    ("成本倒挂（绝对值）", 绝对值倒挂, "无，原值比较的 155 行是符号假象"),
    ("完全重复行", df.duplicated(keep="first"), "业务确认后去重，影响 57 万"),
    ("主单位 = arr", df["主单位"] == "arr", "置空"),
    ("产品类型 = 科研产品2", df["产品类型"] == "科研产品2", "并入科研产品"),
    ("四/五级分类 = 无分类", df["五级分类"] == "无分类", "置空"),
]

pd.DataFrame([{"异常项": k, "行数": int(v.sum()), "占比": f"{v.mean():.2%}", "处置": d}
              for k, v, d in 项])

,异常项,行数,占比,处置
0,退货行（数量<0）,160,0.38%,单价按系统合并，见一
1,退货且 ebs，单价由购退单价覆盖,90,0.22%,已合并
2,采购单价为空,90,0.22%,合并后有值
3,采购单价为 0,4655,11.12%,赠品/未计价，打标记不删
4,单价为 0 但金额非 0,89,0.21%,合并后转为 0 行
5,合并后单价仍为 0,4566,10.91%,金额也全为 0，无影响
6,采购金额 < 0,156,0.37%,退货行，正常
7,成本倒挂（绝对值）,0,0.00%,无，原值比较的 155 行是符号假象
8,完全重复行,209,0.50%,业务确认后去重，影响 57 万
9,主单位 = arr,4,0.01%,置空


## 十、清洗预演

把第一节的单价合并和第五节的规则串起来，得到 44 列底表。**这一步只在内存里跑，不写 parquet**——
规则确认后再落地到 `src/clean`。

In [11]:
底表 = df.copy()

# 5.1 规范化
底表["出入库日期"] = pd.to_datetime(底表["出入库日期"], errors="coerce")
底表["公司编码"] = 底表["公司编码"].astype(str)
底表["物料编码"] = 底表["物料编码"].astype(str)

# 5.2 合并退货单价
底表["采购单价"] = 底表["采购单价"].mask(取退货价, 底表["购退单价"])
底表["采购不含税单价"] = 底表["采购不含税单价"].mask(取退货价, 底表["购退不含税单价"])

# 5.2 归并
底表["采购方式"] = 底表["采购类型名称"].str.replace("合同", "", regex=False).fillna("未知")
底表["供应商内外属性"] = 底表["财务内部外供应商类型"].map({
    "内部关联单位": "内部", "外部单位": "外部", "外部关联单位": "外部",
}).fillna("未知")
底表["产品类型"] = 底表["产品类型"].replace({"科研产品2": "科研产品"})
底表["主单位"] = 底表["主单位"].replace({"arr": None})
底表["仓库类型"] = 底表["仓库类型"].fillna("未知")

# 5.2 五级分类拆成语义两列
设备类 = 底表["三级分类"].isin(["检测设备", "辅助设备", "计量器具", "备品备件"])
底表["检测方法"] = 底表["五级分类"].where(~设备类).replace({"无分类": None})
底表["设备类别"] = 底表["五级分类"].where(设备类).replace({"无分类": None})

# 5.4 丢弃 12 列
丢弃 = ["分析类型", "出入库类型名称", "年份", "月份", "批次效期", "公司简称",
        "多组织名称", "是否多组织内交易", "品牌", "生产地址", "购退单价", "购退不含税单价"]
底表 = 底表.drop(columns=丢弃)

# 6 断言
calc = 底表["采购单价"].fillna(0) * 底表["数量"].fillna(0) / 10000
勾稽 = (calc - 底表["采购金额(万)"].fillna(0)).abs().le(0.01)
print(f"列数：51 → {底表.shape[1]}（目标 44）")
print(f"行数：{len(df):,} → {len(底表):,}")
print(f"断言 · 金额勾稽：{勾稽.sum():,}/{len(底表):,} = {勾稽.mean():.2%}")
print(f"断言 · 行数守恒：{'通过' if len(df) == len(底表) else '失败'}")
print(f"派生列检查：{[c for c in ['采购方式','是否退货','供应商内外属性','检测方法','设备类别'] if c in 底表.columns]}")

底表.dtypes.to_frame("类型")

列数：51 → 44（目标 44）
行数：41,849 → 41,849
断言 · 金额勾稽：41,849/41,849 = 100.00%
断言 · 行数守恒：通过
派生列检查：['采购方式', '是否退货', '供应商内外属性', '检测方法', '设备类别']


,类型
数据来源系统,str
公司编码,str
公司名称,str
板块,str
大区,str
仓库编码,str
仓库名称,str
仓库类型,str
单据编码,str
出入库日期,datetime64[us]
